# 04 — Team-embedding experiments ([HE])

The neural take on the same question: can learned team vectors beat explicit features?
Numeric features come from the SAME M7 matrix (same time split — comparable numbers).
Leakage law, neural edition: the team vocabulary is built from TRAIN only; unseen test
teams map to index 0 (a real `<unk>` vector).

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from cs2analytics.features.matrix import build_feature_matrix
from cs2analytics.models.train_dl import encode_teams, train_model

REPO = Path.cwd()
torch.manual_seed(42)

fs = build_feature_matrix(REPO / "outputs" / "features_v1.parquet")
series = pd.read_csv(REPO / "outputs" / "series_clean.csv")
series["datetime"] = pd.to_datetime(series["datetime"], utc=True, format="ISO8601")
series = series.sort_values("datetime", kind="mergesort").reset_index(drop=True)

pairs = series[["match_id", "team1", "team2"]]
tr_mask = fs.train_mask
# numeric features: drop tier one-hots (keep dense numeric only) - emb net gets teams + numbers
num_names = ["elo_diff", "form5_diff", "rest_days_diff", "is_bo1", "h2h_t1_win_share"]
feat = pd.read_parquet(REPO / "outputs" / "features_v1.parquet")
feat["datetime"] = pd.to_datetime(feat["datetime"], utc=True)
X_num_all = feat[num_names].to_numpy(dtype=np.float32)
y_all = feat["result"].to_numpy(dtype=np.float32)
tr_mask = (feat["datetime"] < pd.Timestamp("2026-01-01", tz="UTC")).to_numpy()

tr_pairs = pairs[tr_mask].reset_index(drop=True)
te_pairs = pairs[~tr_mask].reset_index(drop=True)
idx_tr, idx_te, team_to_idx = encode_teams(tr_pairs, te_pairs)
num_tr, num_te = X_num_all[tr_mask], X_num_all[~tr_mask]
y_tr, y_te = y_all[tr_mask], y_all[~tr_mask]
print(f"train: {len(y_tr)} | test: {len(y_te)} | teams in vocab: {len(team_to_idx) - 1}")

train: 8109 | test: 1811 | teams in vocab: 680


In [2]:
res = train_model(idx_tr, num_tr, y_tr, idx_te, num_te, y_te, emb_dim=16, hidden=32)
print({k: round(v, 4) if isinstance(v, float) else "..." for k, v in res.items() if k != "model"})
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(res["history"], marker="o", ms=3)
ax.set_xlabel("epoch")
ax.set_ylabel("train BCE loss")
ax.set_title("Training curve (early stop on plateau)")
fig.tight_layout()
fig.savefig(REPO / "outputs" / "fig_dl_loss_curve.png", dpi=150)
plt.close(fig)

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\src\cs2analytics\models\train_dl.py:88: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  epoch_loss += float(loss) * len(take)


{'history': '...', 'logloss': 0.6632, 'brier': 0.2349, 'acc': 0.6052, 'train_probs': '...', 'test_probs': '...'}


## 2. Extend the comparison table

In [3]:
comp = pd.read_csv(REPO / "outputs" / "m7_model_comparison.csv")
dl_row = pd.DataFrame(
    [
        {
            "model": "dl_embedding",
            "logloss": res["logloss"],
            "brier": res["brier"],
            "acc": res["acc"],
            "ece": np.nan,
        }
    ]
)
m8 = pd.concat([comp, dl_row], ignore_index=True)
m8.to_csv(REPO / "outputs" / "m8_model_comparison.csv", index=False)
m8

,model,logloss,brier,acc,ece
0,lr,0.646415,0.227812,0.618995,0.024217
1,elo_k32,0.647192,0.228187,0.617891,0.025146
2,gbm,0.651119,0.229754,0.621756,0.025546
3,gbm_isotonic,0.656003,0.232076,0.610160,0.019045
4,constant_0.5,0.693147,0.250000,0.572060,0.072060
5,dl_embedding,0.663182,0.234892,0.605191,NaN


## 3. Embedding sanity — do rivals land close together?

Cosine similarity between the two most-met teams' embeddings vs their actual h2h record.
Prediction: similarity should NOT simply track rivalry — embeddings encode *strength
similarity* (teams that play similar schedules get similar vectors), not head-to-head
dynamics. Report both numbers either way.

In [4]:
from cs2analytics.features.form import h2h_record

E = res["model"].embedding.weight.detach().numpy()
idx_to_team = {v: k for k, v in team_to_idx.items() if v != 0}

pairs_counts = (
    series.groupby([series["team1"].str.strip(), series["team2"].str.strip()])
    .size()
    .unstack(fill_value=0)
)
meetings = {}
for _, r in series.iterrows():
    key = tuple(sorted([r["team1"], r["team2"]]))
    meetings[key] = meetings.get(key, 0) + 1
top_pairs = sorted(meetings.items(), key=lambda kv: -kv[1])[:3]
for (a, b), n_meet in top_pairs:
    ia, ib = team_to_idx.get(a), team_to_idx.get(b)
    if ia is None or ib is None or ia == 0 or ib == 0:
        continue
    va, vb = E[ia], E[ib]
    cos = float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))
    aw, bw = h2h_record(series, a, b, before_ts=series["datetime"].max())
    print(
        f"{a} vs {b}: meetings={n_meet}, h2h={aw}-{bw} ({aw / (aw + bw):.2f} a-share), cosine={cos:.3f}"
    )

MOUZ vs Team Vitality: meetings=18, h2h=4-14 (0.22 a-share), cosine=0.012
FaZe Clan vs MOUZ: meetings=17, h2h=11-6 (0.65 a-share), cosine=0.219
MOUZ vs Team Spirit: meetings=16, h2h=5-11 (0.31 a-share), cosine=0.299


## 4. Conclusion (honest either way)

Fill from the numbers above: if the MLP's logloss lands ABOVE LR's 0.6464, the finding is
that at n≈8k series with 4-5 informative numeric features, a 2-layer net with team
embeddings cannot beat a well-specified logistic model — the signal is linear-ish in Elo
space and the embedding adds variance, not information. If it lands below, the embedding
captures matchup non-linearity the diff-features miss. Either conclusion is a result.